In [1]:
# Import libraries, configure display settings, and initialize BigQuery client.
import warnings

import pandas as pd
from google.cloud import bigquery
import pybaseball

PROJECT_ID = "baseball-analytics-portfolio"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

warnings.filterwarnings("ignore", module="pybaseball")

client = bigquery.Client(project=PROJECT_ID)
print("Environment ready")


Environment ready


# Baseball Analytics - Data Profiling Notebook
Quick reference for exploring raw, staging, and mart tables in BigQuery.  
All queries run against baseball-analytics-portfolio.


In [2]:
# Define reusable helpers for running SQL and previewing BigQuery tables.
def run_query(sql: str) -> pd.DataFrame:
    """Run a BigQuery SQL query and return results as a DataFrame."""
    return client.query(sql).result().to_dataframe()


def preview_table(dataset: str, table: str, limit: int = 10) -> pd.DataFrame:
    """Preview first N rows of any BigQuery table."""
    sql = f"SELECT * FROM `{PROJECT_ID}.{dataset}.{table}` LIMIT {int(limit)}"
    return run_query(sql)


## Raw Layer - statcast_pitches


In [3]:
# Summarize raw Statcast table coverage and pitch-type distribution.
raw_summary_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT DATE(TIMESTAMP_SECONDS(SAFE_CAST(game_date / 1000000000 AS INT64)))) AS distinct_game_dates,
  MIN(DATE(TIMESTAMP_SECONDS(SAFE_CAST(game_date / 1000000000 AS INT64)))) AS min_game_date,
  MAX(DATE(TIMESTAMP_SECONDS(SAFE_CAST(game_date / 1000000000 AS INT64)))) AS max_game_date,
  COUNT(DISTINCT pitcher) AS distinct_pitchers,
  COUNT(DISTINCT batter) AS distinct_batters
FROM `{PROJECT_ID}.raw.statcast_pitches`
"""

raw_pitch_type_sql = f"""
SELECT
  pitch_type,
  COUNT(*) AS pitch_count
FROM `{PROJECT_ID}.raw.statcast_pitches`
GROUP BY pitch_type
ORDER BY pitch_count DESC
"""

raw_summary_df = run_query(raw_summary_sql)
raw_pitch_type_df = run_query(raw_pitch_type_sql)

print("Raw table summary:")
display(raw_summary_df)
print("Distinct pitch types with counts:")
display(raw_pitch_type_df)


Raw table summary:


,total_rows,distinct_game_dates,min_game_date,max_game_date,distinct_pitchers,distinct_batters
0,258154,66,2026-03-27,2026-05-31,654,530


Distinct pitch types with counts:


,pitch_type,pitch_count
0,FF,78184
1,SI,42737
2,SL,34613
3,CH,28905
4,ST,21043
5,FC,19858
6,CU,16867
7,FS,8584
8,KC,4313
9,SV,1276


In [4]:
# Preview a small sample of rows from the raw statcast table.
preview_table("raw", "statcast_pitches", limit=5)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,SI,1775088000000000000,93.400,-2.010,5.710,"Bido, Osvaldo",694374,674370,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,4,Tim Tawa pops out to first baseman Matt Olson.,R,R,R,AZ,ATL,X,3,popup,2,1,2026,-1.230,0.680,-0.478,2.596,<NA>,545121,<NA>,2,9,Bot,167.650,148.910,<NA>,<NA>,<NA>,<NA>,6.652,-135.796,-3.520,-16.677,32.263,-23.239,3.163,1.596,166,62.100,51,93.300,2265,6.500,825105,686948,621566,645277,571657,643289,573262,671739,642201,54.030,0.054,0.049,0.000,1,0,0,3,87,4,Sinker,2,17,2,17,17,2,2,17,Standard,Standard,219,-0.001,-0.280,70.900,6.600,0.060,0.280,88.000,-15,-15,0.001,0.001,30,27,31,27,2,0,5,1,4,1,1.960,1.230,1.230,32.900,-2.344,3.872,30.056,28.849,24.266
1,SI,1775088000000000000,95.400,-2.160,5.640,"Bido, Osvaldo",694374,674370,None,foul,<NA>,<NA>,<NA>,<NA>,2,None,R,R,R,AZ,ATL,S,<NA>,None,2,0,2026,-1.250,0.420,0.073,2.690,<NA>,545121,<NA>,2,9,Bot,NaN,NaN,<NA>,<NA>,<NA>,<NA>,8.653,-138.688,-2.854,-18.128,32.973,-26.267,3.163,1.596,<NA>,NaN,<NA>,95.200,2367,6.400,825105,686948,621566,645277,571657,643289,573262,671739,642201,54.130,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,87,3,Sinker,2,17,2,17,17,2,2,17,Standard,Standard,217,0.000,-0.040,76.400,7.100,NaN,0.040,NaN,-15,-15,0.001,0.001,30,27,31,27,2,0,5,1,4,1,2.100,1.250,1.250,30.900,8.808,-0.952,21.994,35.309,23.796
2,SI,1775088000000000000,94.600,-2.030,5.710,"Bido, Osvaldo",694374,674370,None,ball,<NA>,<NA>,<NA>,<NA>,14,None,R,R,R,AZ,ATL,B,<NA>,None,1,0,2026,-1.310,0.590,0.595,1.092,<NA>,545121,<NA>,2,9,Bot,NaN,NaN,<NA>,<NA>,<NA>,<NA>,9.740,-137.319,-7.445,-18.681,29.656,-23.174,3.163,1.596,<NA>,NaN,<NA>,94.800,2354,6.500,825105,686948,621566,645277,571657,643289,573262,671739,642201,54.030,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,87,2,Sinker,2,17,2,17,17,2,2,17,Standard,Standard,224,0.000,0.028,NaN,NaN,NaN,-0.028,NaN,-15,-15,0.001,0.001,30,27,31,27,2,0,5,1,4,1,1.960,1.310,1.310,31.300,NaN,NaN,NaN,NaN,NaN
3,FF,1775088000000000000,94.400,-1.930,5.820,"Bido, Osvaldo",694374,674370,None,ball,<NA>,<NA>,<NA>,<NA>,12,None,R,R,R,AZ,ATL,B,<NA>,None,0,0,2026,-0.470,1.360,1.017,3.382,<NA>,545121,<NA>,2,9,Bot,NaN,NaN,<NA>,<NA>,<NA>,<NA>,8.634,-137.170,-3.529,-7.824,30.549,-14.443,3.163,1.596,<NA>,NaN,<NA>,94.500,2464,6.400,825105,686948,621566,645277,571657,643289,573262,671739,642201,54.060,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,87,1,4-Seam Fastball,2,17,2,17,17,2,2,17,Standard,Standard,217,0.000,0.026,NaN,NaN,NaN,-0.026,NaN,-15,-15,0.001,0.001,30,27,31,27,2,

## Staging Layer - stg_statcast_pitches


In [5]:
# Validate staging row counts, null checks, date range, and outcome distribution.
stg_summary_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(pitch_type IS NULL) AS null_pitch_type_rows,
  COUNTIF(game_date IS NULL) AS null_game_date_rows,
  MIN(game_date) AS min_game_date,
  MAX(game_date) AS max_game_date
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
"""

stg_outcome_sql = f"""
WITH base AS (
  SELECT pitch_outcome_category
  FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
), totals AS (
  SELECT COUNT(*) AS total_rows FROM base
)
SELECT
  b.pitch_outcome_category,
  COUNT(*) AS outcome_count,
  SAFE_DIVIDE(COUNT(*), t.total_rows) AS outcome_pct
FROM base b
CROSS JOIN totals t
GROUP BY b.pitch_outcome_category, t.total_rows
ORDER BY outcome_count DESC
"""

stg_summary_df = run_query(stg_summary_sql)
stg_outcome_df = run_query(stg_outcome_sql)

print("Staging table checks:")
display(stg_summary_df)
print("pitch_outcome_category distribution (counts and percentages):")
display(stg_outcome_df)


Staging table checks:


,total_rows,null_pitch_type_rows,null_game_date_rows,min_game_date,max_game_date
0,257220,0,0,2026-03-27,2026-05-31


pitch_outcome_category distribution (counts and percentages):


,pitch_outcome_category,outcome_count,outcome_pct
0,S,118674,0.461
1,B,93837,0.365
2,X,44709,0.174


In [6]:
# Profile key numeric columns with non-null count, average, min, max, and stddev.
stg_numeric_profile_sql = f"""
SELECT 'pitch_velocity_mph' AS column_name, COUNT(pitch_velocity_mph) AS non_null_count, AVG(pitch_velocity_mph) AS avg_value, MIN(pitch_velocity_mph) AS min_value, MAX(pitch_velocity_mph) AS max_value, STDDEV(pitch_velocity_mph) AS stddev_value
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
UNION ALL
SELECT 'spin_rate_rpm', COUNT(spin_rate_rpm), AVG(spin_rate_rpm), MIN(spin_rate_rpm), MAX(spin_rate_rpm), STDDEV(spin_rate_rpm)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
UNION ALL
SELECT 'exit_velocity_mph', COUNT(exit_velocity_mph), AVG(exit_velocity_mph), MIN(exit_velocity_mph), MAX(exit_velocity_mph), STDDEV(exit_velocity_mph)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
UNION ALL
SELECT 'launch_angle_deg', COUNT(launch_angle_deg), AVG(launch_angle_deg), MIN(launch_angle_deg), MAX(launch_angle_deg), STDDEV(launch_angle_deg)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
UNION ALL
SELECT 'xba', COUNT(xba), AVG(xba), MIN(xba), MAX(xba), STDDEV(xba)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
UNION ALL
SELECT 'xwoba', COUNT(xwoba), AVG(xwoba), MIN(xwoba), MAX(xwoba), STDDEV(xwoba)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
UNION ALL
SELECT 'bat_speed', COUNT(bat_speed), AVG(bat_speed), MIN(bat_speed), MAX(bat_speed), STDDEV(bat_speed)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
UNION ALL
SELECT 'swing_length', COUNT(swing_length), AVG(swing_length), MIN(swing_length), MAX(swing_length), STDDEV(swing_length)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
ORDER BY column_name
"""

stg_numeric_profile_df = run_query(stg_numeric_profile_sql)
stg_numeric_profile_df


,column_name,non_null_count,avg_value,min_value,max_value,stddev_value
0,bat_speed,117635,69.774,1.200,87.900,9.463
1,exit_velocity_mph,84048,82.712,3.900,119.000,15.449
2,launch_angle_deg,84162,17.932,-89.000,90.000,32.799
3,pitch_velocity_mph,257220,89.467,30.300,103.800,6.117
4,spin_rate_rpm,256290,2260.945,19.000,3599.000,371.464
5,swing_length,117635,7.239,0.300,26.600,1.064
6,xba,43773,0.326,0.001,1.000,0.294
7,xwoba,65462,0.320,0.000,2.044,0.375


In [7]:
# Analyze pitch-type distribution with average velocity, spin, and xBA.
pitch_type_distribution_sql = f"""
SELECT
  pitch_type,
  pitch_name,
  COUNT(*) AS pitch_count,
  AVG(pitch_velocity_mph) AS avg_pitch_velocity_mph,
  AVG(spin_rate_rpm) AS avg_spin_rate_rpm,
  AVG(xba) AS avg_xba
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
GROUP BY pitch_type, pitch_name
ORDER BY pitch_count DESC
"""

pitch_type_distribution_df = run_query(pitch_type_distribution_sql)
pitch_type_distribution_df


,pitch_type,pitch_name,pitch_count,avg_pitch_velocity_mph,avg_spin_rate_rpm,avg_xba
0,FF,4-Seam Fastball,78184,94.641,2312.936,0.331
1,SI,Sinker,42737,93.944,2188.282,0.335
2,SL,Slider,34613,86.088,2430.543,0.323
3,CH,Changeup,28905,86.070,1743.780,0.315
4,ST,Sweeper,21043,82.829,2593.424,0.303
5,FC,Cutter,19858,89.632,2387.875,0.330
6,CU,Curveball,16867,79.973,2585.841,0.325
7,FS,Split-Finger,8584,86.609,1391.815,0.310
8,KC,Knuckle Curve,4313,82.382,2518.924,0.345
9,SV,Slurve,1276,81.926,2562.635,0.347


## Mart Layer - Batter Game Stats


In [8]:
# Summarize batter mart volume and key rate metrics across all rows.
mart_batter_overview_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT batter_id) AS distinct_batters,
  COUNT(DISTINCT game_id) AS distinct_games,
  AVG(hard_hit_rate) AS avg_hard_hit_rate,
  AVG(barrel_rate) AS avg_barrel_rate,
  AVG(avg_xwoba) AS avg_avg_xwoba,
  AVG(avg_bat_speed) AS avg_avg_bat_speed
FROM `{PROJECT_ID}.marts.mart_batter_game_stats`
"""

mart_batter_overview_df = run_query(mart_batter_overview_sql)
print("Batter mart overview:")
display(mart_batter_overview_df)
print("Note: avg_bat_speed is expected to be mostly null for older seasons.")


Batter mart overview:


,total_rows,distinct_batters,distinct_games,avg_hard_hit_rate,avg_barrel_rate,avg_avg_xwoba,avg_avg_bat_speed
0,17778,530,875,0.389,0.080,0.312,69.764


Note: avg_bat_speed is expected to be mostly null for older seasons.


In [9]:
# Find top batters by average exit velocity with at least 5 total batted balls.
top_batters_sql = f"""
SELECT
  batter_id,
  COUNT(*) AS games,
  SUM(batted_balls) AS total_batted_balls,
  AVG(avg_exit_velocity_mph) AS avg_exit_velocity_mph,
  AVG(avg_xwoba) AS avg_xwoba,
  SAFE_DIVIDE(SUM(hard_hit_count), SUM(batted_balls)) AS avg_hard_hit_rate,
  SUM(home_runs) AS total_home_runs
FROM `{PROJECT_ID}.marts.mart_batter_game_stats`
GROUP BY batter_id
HAVING SUM(batted_balls) >= 5
ORDER BY avg_exit_velocity_mph DESC
LIMIT 20
"""

top_batters_df = run_query(top_batters_sql)
top_batters_df


,batter_id,games,total_batted_balls,avg_exit_velocity_mph,avg_xwoba,avg_hard_hit_rate,total_home_runs
0,621550,5,8,98.583,0.207,0.750,0
1,682987,10,12,96.960,0.252,0.750,0
2,672356,9,15,96.947,0.307,0.467,2
3,695578,59,146,96.637,0.446,0.610,16
4,665833,57,142,96.333,0.342,0.613,13
5,669899,3,6,96.240,0.448,0.167,1
6,672284,19,33,95.550,0.342,0.545,1
7,670541,59,168,95.170,0.497,0.542,20
8,808959,56,119,94.990,0.356,0.580,19
9,669398,16,22,94.571,0.254,0.455,2


## Mart Layer - Pitcher Game Stats


In [10]:
# Summarize pitcher mart volume and core performance metrics.
mart_pitcher_overview_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT pitcher_id) AS distinct_pitchers,
  COUNT(DISTINCT game_id) AS distinct_games,
  AVG(whiff_rate) AS avg_whiff_rate,
  AVG(strike_pct) AS avg_strike_pct,
  AVG(avg_pitch_velocity_mph) AS avg_avg_pitch_velocity_mph
FROM `{PROJECT_ID}.marts.mart_pitcher_game_stats`
"""

mart_pitcher_overview_df = run_query(mart_pitcher_overview_sql)
print("Pitcher mart overview:")
display(mart_pitcher_overview_df)


Pitcher mart overview:


,total_rows,distinct_pitchers,distinct_games,avg_whiff_rate,avg_strike_pct,avg_avg_pitch_velocity_mph
0,7389,654,875,0.233,0.461,89.556


In [11]:
# Find top pitchers by whiff rate with at least 50 total pitches.
top_pitchers_sql = f"""
WITH agg AS (
  SELECT
    pitcher_id,
    pitcher_name,
    COUNT(*) AS games,
    SUM(total_pitches) AS total_pitches,
    AVG(avg_pitch_velocity_mph) AS avg_velocity,
    SAFE_DIVIDE(SUM(whiffs), SUM(swings)) AS avg_whiff_rate,
    SAFE_DIVIDE(SUM(strikes), SUM(total_pitches)) AS avg_strike_pct,
    SAFE_DIVIDE(SUM(hard_hit_allowed_count), SUM(batted_balls_allowed)) AS avg_hard_hit_allowed_rate
  FROM `{PROJECT_ID}.marts.mart_pitcher_game_stats`
  GROUP BY pitcher_id, pitcher_name
)
SELECT *
FROM agg
WHERE total_pitches >= 50
ORDER BY avg_whiff_rate DESC
LIMIT 20
"""

top_pitchers_df = run_query(top_pitchers_sql)
top_pitchers_df

,pitcher_id,pitcher_name,games,total_pitches,avg_velocity,avg_whiff_rate,avg_strike_pct,avg_hard_hit_allowed_rate
0,695243,"Miller, Mason",24,413,93.829,0.500,0.574,0.139
1,672021,"Cerantola, Eric",3,79,88.479,0.484,0.443,0.714
2,605218,"Edwards Jr., Carl",2,99,88.016,0.419,0.525,0.400
3,682610,"Muñoz, Roddery",3,112,90.376,0.413,0.429,0.769
4,676130,"Buttó, José",3,63,90.788,0.409,0.365,0.429
5,662253,"Muñoz, Andrés",23,381,91.233,0.405,0.501,0.463
6,605483,"Snell, Blake",1,77,89.516,0.400,0.506,0.182
7,518585,"Cruz, Fernando",28,408,84.751,0.399,0.468,0.259
8,673513,"Matsui, Yuki",8,214,87.236,0.386,0.467,0.294
9,682825,"Mey, Luis",6,171,93.843,0.379,0.450,0.412


In [12]:
# Show pitch-mix percentages for high-volume pitchers.
pitch_mix_sql = f"""
WITH agg AS (
  SELECT
    pitcher_id,
    ANY_VALUE(pitcher_name) AS pitcher_name,
    SUM(total_pitches) AS total_pitches,
    AVG(pct_four_seam) AS pct_four_seam,
    AVG(pct_sinker) AS pct_sinker,
    AVG(pct_cutter) AS pct_cutter,
    AVG(pct_slider) AS pct_slider,
    AVG(pct_sweeper) AS pct_sweeper,
    AVG(pct_curveball) AS pct_curveball,
    AVG(pct_changeup) AS pct_changeup,
    AVG(pct_splitter) AS pct_splitter,
    AVG(pct_other) AS pct_other
  FROM `{PROJECT_ID}.marts.mart_pitcher_game_stats`
  GROUP BY pitcher_id
)
SELECT
  pitcher_name,
  total_pitches,
  pct_four_seam,
  pct_sinker,
  pct_cutter,
  pct_slider,
  pct_sweeper,
  pct_curveball,
  pct_changeup,
  pct_splitter,
  pct_other
FROM agg
WHERE total_pitches >= 50
ORDER BY total_pitches DESC
LIMIT 20
"""

pitch_mix_df = run_query(pitch_mix_sql)
pitch_mix_df


,pitcher_name,total_pitches,pct_four_seam,pct_sinker,pct_cutter,pct_slider,pct_sweeper,pct_curveball,pct_changeup,pct_splitter,pct_other
0,"Alcantara, Sandy",1155,0.200,0.226,0.132,0.108,0.070,0.044,0.219,0.000,0.000
1,"Luzardo, Jesús",1153,0.277,0.138,0.000,0.000,0.358,0.000,0.227,0.000,0.000
2,"Detmers, Reid",1149,0.439,0.025,0.000,0.309,0.000,0.105,0.121,0.000,0.001
3,"Ray, Robbie",1140,0.438,0.029,0.000,0.288,0.000,0.000,0.164,0.000,0.081
4,"Leiter, Jack",1126,0.379,0.077,0.104,0.178,0.000,0.079,0.182,0.000,0.000
5,"Bradish, Kyle",1122,0.194,0.305,0.000,0.273,0.000,0.228,0.000,0.000,0.000
6,"Lugo, Seth",1120,0.168,0.199,0.153,0.097,0.098,0.150,0.082,0.000,0.053
7,"Williams, Gavin",1113,0.270,0.159,0.088,0.000,0.259,0.224,0.000,0.000,0.000
8,"Griffin, Foster",1110,0.178,0.109,0.311,0.000,0.152,0.086,0.108,0.056,0.000
9,"Gausman, Kevin",1106,0.532,0.000,0.000,0.078,0.000,0.000,0.000,0.390,0.000


## Player Lookup - Mapping IDs to Names


In [13]:
# Build a player lookup table from unique batter and pitcher MLBAM IDs.
batter_ids_sql = f"""
SELECT DISTINCT CAST(batter_id AS INT64) AS player_id
FROM `{PROJECT_ID}.marts.mart_batter_game_stats`
WHERE batter_id IS NOT NULL
"""

pitcher_ids_sql = f"""
SELECT DISTINCT CAST(pitcher_id AS INT64) AS player_id
FROM `{PROJECT_ID}.marts.mart_pitcher_game_stats`
WHERE pitcher_id IS NOT NULL
"""

batter_ids = run_query(batter_ids_sql)["player_id"].dropna().astype(int).tolist()
pitcher_ids = run_query(pitcher_ids_sql)["player_id"].dropna().astype(int).tolist()
all_player_ids = sorted(set(batter_ids + pitcher_ids))

if all_player_ids:
    player_lookup = pybaseball.playerid_reverse_lookup(all_player_ids, key_type="mlbam")
else:
    player_lookup = pd.DataFrame()

print(f"player_lookup shape: {player_lookup.shape}")
display(player_lookup.head(10))
print("player_lookup columns:")
print(player_lookup.columns.tolist())


Gathering player lookup table. This may take a moment.


player_lookup shape: (1159, 8)


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,schneider,davis,676914,schnd001,schneda03,23565,2023.000,2026.000
1,durbin,caleb,702332,durbc002,durbica01,29646,2025.000,2026.000
2,granillo,andre,701552,grana001,granian01,29542,2025.000,2026.000
3,díaz,edwin,621242,diaze006,diazed04,14710,2016.000,2026.000
4,lopez,nicky,670032,lopen001,lopezni01,19339,2019.000,2026.000
5,lowe,brandon,664040,loweb001,lowebr01,18882,2018.000,2026.000
6,junis,jakob,596001,junij001,junisja01,13619,2017.000,2026.000
7,goodman,hunter,696100,goodh001,goodmhu01,29715,2023.000,2026.000
8,giolito,lucas,608337,gioll001,giolilu01,15474,2016.000,2026.000
9,assad,javier,665871,assaj001,assadja01,21741,2022.000,2026.000


player_lookup columns:
['name_last', 'name_first', 'key_mlbam', 'key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last']


In [14]:
# Merge player names onto top batter results and show ranked output.
lookup_id_col = "key_mlbam" if "key_mlbam" in player_lookup.columns else "player_id"

batter_lookup_df = player_lookup.copy()
if lookup_id_col in batter_lookup_df.columns:
    batter_lookup_df[lookup_id_col] = pd.to_numeric(batter_lookup_df[lookup_id_col], errors="coerce")

top_batters_named_df = top_batters_df.copy()
top_batters_named_df["batter_id"] = pd.to_numeric(top_batters_named_df["batter_id"], errors="coerce")

top_batters_named_df = top_batters_named_df.merge(
    batter_lookup_df,
    how="left",
    left_on="batter_id",
    right_on=lookup_id_col
)

top_batters_named_df = top_batters_named_df[[
    "name_first",
    "name_last",
    "batter_id",
    "games",
    "avg_exit_velocity_mph",
    "avg_xwoba",
    "avg_hard_hit_rate",
    "total_home_runs"
]].sort_values("avg_exit_velocity_mph", ascending=False)

top_batters_named_df


,name_first,name_last,batter_id,games,avg_exit_velocity_mph,avg_xwoba,avg_hard_hit_rate,total_home_runs
0,patrick,wisdom,621550,5,98.583,0.207,0.750,0
1,spencer,jones,682987,10,96.960,0.252,0.750,0
2,gabriel,arias,672356,9,96.947,0.307,0.467,2
3,james,wood,695578,59,96.637,0.446,0.610,16
4,oneil,cruz,665833,57,96.333,0.342,0.613,13
5,ryan,ward,669899,3,96.240,0.448,0.167,1
6,jarred,kelenic,672284,19,95.550,0.342,0.545,1
7,yordan,álvarez,670541,59,95.170,0.497,0.542,20
8,munetaka,murakami,808959,56,94.990,0.356,0.580,19
9,gage,workman,669398,16,94.571,0.254,0.455,2


In [15]:
# Merge player names onto top pitcher results and show ranked output.
lookup_id_col = "key_mlbam" if "key_mlbam" in player_lookup.columns else "player_id"

pitcher_lookup_df = player_lookup.copy()
if lookup_id_col in pitcher_lookup_df.columns:
    pitcher_lookup_df[lookup_id_col] = pd.to_numeric(pitcher_lookup_df[lookup_id_col], errors="coerce")

top_pitchers_named_df = top_pitchers_df.copy()
top_pitchers_named_df["pitcher_id"] = pd.to_numeric(top_pitchers_named_df["pitcher_id"], errors="coerce")

top_pitchers_named_df = top_pitchers_named_df.merge(
    pitcher_lookup_df,
    how="left",
    left_on="pitcher_id",
    right_on=lookup_id_col
)

top_pitchers_named_df = top_pitchers_named_df[[
    "name_first",
    "name_last",
    "pitcher_id",
    "games",
    "total_pitches",
    "avg_velocity",
    "avg_whiff_rate",
    "avg_hard_hit_allowed_rate"
]].sort_values("avg_whiff_rate", ascending=False)

top_pitchers_named_df


,name_first,name_last,pitcher_id,games,total_pitches,avg_velocity,avg_whiff_rate,avg_hard_hit_allowed_rate
0,mason,miller,695243,24,413,93.829,0.500,0.139
1,eric,cerantola,672021,3,79,88.479,0.484,0.714
2,carl,edwards,605218,2,99,88.016,0.419,0.400
3,roddery,muñoz,682610,3,112,90.376,0.413,0.769
4,josé,buttó,676130,3,63,90.788,0.409,0.429
5,andrés,muñoz,662253,23,381,91.233,0.405,0.463
6,blake,snell,605483,1,77,89.516,0.400,0.182
7,fernando,cruz,518585,28,408,84.751,0.399,0.259
8,yuki,matsui,673513,8,214,87.236,0.386,0.294
9,luis,mey,682825,6,171,93.843,0.379,0.412


## Data Quality Checks


In [16]:
# Calculate null rates for selected staging columns.
null_rate_sql = f"""
WITH total AS (
  SELECT COUNT(*) AS total_rows
  FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
)
SELECT 'exit_velocity_mph' AS column_name, COUNTIF(exit_velocity_mph IS NULL) AS null_count, t.total_rows, SAFE_DIVIDE(COUNTIF(exit_velocity_mph IS NULL), t.total_rows) AS null_pct
FROM `{PROJECT_ID}.staging.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'launch_angle_deg', COUNTIF(launch_angle_deg IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(launch_angle_deg IS NULL), t.total_rows)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'xba', COUNTIF(xba IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(xba IS NULL), t.total_rows)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'xwoba', COUNTIF(xwoba IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(xwoba IS NULL), t.total_rows)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'xslg', COUNTIF(xslg IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(xslg IS NULL), t.total_rows)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'bat_speed', COUNTIF(bat_speed IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(bat_speed IS NULL), t.total_rows)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'swing_length', COUNTIF(swing_length IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(swing_length IS NULL), t.total_rows)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'spin_rate_rpm', COUNTIF(spin_rate_rpm IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(spin_rate_rpm IS NULL), t.total_rows)
FROM `{PROJECT_ID}.staging.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
ORDER BY null_pct DESC
"""

null_rate_df = run_query(null_rate_sql)
null_rate_df


,column_name,null_count,total_rows,null_pct
0,xba,213447,257220,0.830
1,xslg,213447,257220,0.830
2,xwoba,191758,257220,0.746
3,exit_velocity_mph,173172,257220,0.673
4,launch_angle_deg,173058,257220,0.673
5,bat_speed,139585,257220,0.543
6,swing_length,139585,257220,0.543
7,spin_rate_rpm,930,257220,0.004


In [17]:
# Show pitch volume by game date to verify coverage and identify potential gaps.
game_date_distribution_sql = f"""
SELECT
  game_date,
  COUNT(*) AS pitch_count
FROM `{PROJECT_ID}.staging.stg_statcast_pitches`
GROUP BY game_date
ORDER BY game_date
"""

game_date_distribution_df = run_query(game_date_distribution_sql)
game_date_distribution_df


,game_date,pitch_count
0,2026-03-27,2246
1,2026-03-28,4759
2,2026-03-29,3647
3,2026-03-30,4333
4,2026-03-31,3995
5,2026-04-01,4425
6,2026-04-02,920
7,2026-04-03,4142
8,2026-04-04,4601
9,2026-04-05,5028


## Team Dimension & Coverage  _(added 2026-06-21)_
Validate the new `dim_team` + `team_*` columns now attached to the reporting
models. Sources: `marts.dim_team`, `reporting.rpt_batter_season`,
`reporting.rpt_pitcher_season`, `staging.stg_mlb_rosters`.

In [18]:
# Team match rate on the season reporting models.
# Expect ~492/530 batters and ~594/654 pitchers (unmatched = not on a 40-man).
team_coverage_sql = f"""
SELECT 'batter' AS role,
  COUNT(*) AS total_players,
  COUNTIF(team_id IS NOT NULL) AS with_team,
  SAFE_DIVIDE(COUNTIF(team_id IS NOT NULL), COUNT(*)) AS match_rate
FROM `{PROJECT_ID}.reporting.rpt_batter_season`
UNION ALL
SELECT 'pitcher',
  COUNT(*),
  COUNTIF(team_id IS NOT NULL),
  SAFE_DIVIDE(COUNTIF(team_id IS NOT NULL), COUNT(*))
FROM `{PROJECT_ID}.reporting.rpt_pitcher_season`
"""
team_coverage_df = run_query(team_coverage_sql)
print("Team match rate (expect ~492/530 batters, ~594/654 pitchers):")
display(team_coverage_df)

Team match rate (expect ~492/530 batters, ~594/654 pitchers):


,role,total_players,with_team,match_rate
0,batter,530,492,0.928
1,pitcher,654,594,0.908


In [19]:
# Join integrity: a NULL team should ONLY happen when the player is not on a
# 40-man roster. If any player has a NULL team yet IS on the 40-man, that is a
# join bug. null_team_but_on_40man must be 0.
team_join_integrity_sql = f"""
SELECT
  COUNTIF(b.team_id IS NULL AND r.mlbam_id IS NOT NULL) AS null_team_but_on_40man,
  COUNTIF(b.team_id IS NULL) AS null_team_total
FROM `{PROJECT_ID}.reporting.rpt_batter_season` b
LEFT JOIN `{PROJECT_ID}.staging.stg_mlb_rosters` r ON r.mlbam_id = b.batter_id
"""
team_join_integrity_df = run_query(team_join_integrity_sql)
display(team_join_integrity_df)
assert int(team_join_integrity_df["null_team_but_on_40man"][0]) == 0, \
    "JOIN BUG: a player has NULL team but IS on the 40-man roster"
print("PASS: every unmatched player is genuinely not on a 40-man roster (not a join bug).")

# A few unmatched batters, by volume, for eyeballing.
unmatched_sample_sql = f"""
SELECT batter_id, batter_name, games, plate_appearances
FROM `{PROJECT_ID}.reporting.rpt_batter_season`
WHERE team_id IS NULL
ORDER BY games DESC
LIMIT 10
"""
print("Sample unmatched batters (no current 40-man team):")
display(run_query(unmatched_sample_sql))

,null_team_but_on_40man,null_team_total
0,0,38


PASS: every unmatched player is genuinely not on a 40-man roster (not a join bug).
Sample unmatched batters (no current 40-man team):


,batter_id,batter_name,games,plate_appearances
0,650859,Luis Rengifo,45,167
1,592206,Nick Castellanos,38,120
2,457705,Andrew McCutchen,36,81
3,664059,Sam Haggerty,26,45
4,596103,Austin Slater,21,49
5,666624,Christopher Morel,20,67
6,669289,Santiago Espinal,20,47
7,687957,Dustin Harris,17,52
8,666464,Jerar Encarnacion,16,35
9,669369,Bryce Johnson,14,36


In [20]:
# Player-with-stats distribution across the 30 teams. Sanity: fairly even,
# no team wildly over/under-represented (would hint at a mapping skew).
team_distribution_sql = f"""
WITH bat AS (
  SELECT team_id, COUNT(*) AS batters
  FROM `{PROJECT_ID}.reporting.rpt_batter_season`
  WHERE team_id IS NOT NULL GROUP BY team_id
),
pit AS (
  SELECT team_id, COUNT(*) AS pitchers
  FROM `{PROJECT_ID}.reporting.rpt_pitcher_season`
  WHERE team_id IS NOT NULL GROUP BY team_id
)
SELECT t.team_abbrev, t.division_name,
  COALESCE(bat.batters, 0) AS batters,
  COALESCE(pit.pitchers, 0) AS pitchers,
  COALESCE(bat.batters, 0) + COALESCE(pit.pitchers, 0) AS players
FROM `{PROJECT_ID}.marts.dim_team` t
LEFT JOIN bat ON bat.team_id = t.team_id
LEFT JOIN pit ON pit.team_id = t.team_id
ORDER BY players DESC
"""
team_distribution_df = run_query(team_distribution_sql)
print(f"Players with stats per team ({len(team_distribution_df)} teams):")
display(team_distribution_df)
print(
    "players/team -> min={}, median={}, max={}".format(
        team_distribution_df["players"].min(),
        int(team_distribution_df["players"].median()),
        team_distribution_df["players"].max(),
    )
)

Players with stats per team (30 teams):


,team_abbrev,division_name,batters,pitchers,players
0,LAD,National League West,16,25,41
1,NYM,National League East,20,20,40
2,BAL,American League East,16,24,40
3,DET,American League Central,18,21,39
4,LAA,American League West,19,19,38
5,MIA,National League East,17,21,38
6,CHC,National League Central,14,24,38
7,CWS,American League Central,16,21,37
8,NYY,American League East,16,21,37
9,BOS,American League East,15,22,37


players/team -> min=32, median=36, max=41


## Roster / Status Dimension  _(added 2026-06-21)_
Validate `dim_roster_status` (the full rostered population, grain `mlbam_id`)
and the `availability` / `status_description` columns attached to the reporting
models. Sources: `marts.dim_roster_status`, `reporting.rpt_*`.

In [21]:
# Availability distribution across the FULL rostered population.
# Expect ~6,512 Active, 585 IL — 60-Day, 368 IL — 7-Day, 158 IL — Full Season,
# etc.; total ~8,189 (8,200 fullRoster rows deduped to one status per player).
status_dist_sql = f"""
SELECT availability,
  COUNT(*) AS players,
  SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()) AS pct
FROM `{PROJECT_ID}.marts.dim_roster_status`
GROUP BY availability
ORDER BY players DESC
"""
status_dist_df = run_query(status_dist_sql)
print("dim_roster_status availability distribution:")
display(status_dist_df)
print(f"total rostered players (grain mlbam_id): {int(status_dist_df['players'].sum())}")

dim_roster_status availability distribution:


,availability,players,pct
0,Active,6512,0.795
1,IL — 60-Day,585,0.071
2,IL — 7-Day,368,0.045
3,Minors,341,0.042
4,IL — Full Season,158,0.019
5,IL — 15-Day,56,0.007
6,IL — 10-Day,55,0.007
7,Rehab Assignment,48,0.006
8,Restricted List,36,0.004
9,Designated for Assignment,13,0.002


total rostered players (grain mlbam_id): 8189


In [22]:
# Status attach coverage on the reporting models. A row with stats but NULL
# availability = the player played but is not currently rostered (released /
# traded out of MLB) -> expected, not a bug. We just quantify it.
status_attach_sql = f"""
SELECT 'rpt_batter_season' AS model, COUNT(*) AS n_rows,
  COUNTIF(availability IS NOT NULL) AS with_availability,
  COUNTIF(status_description IS NOT NULL) AS with_status_desc
FROM `{PROJECT_ID}.reporting.rpt_batter_season`
UNION ALL
SELECT 'rpt_pitcher_season', COUNT(*),
  COUNTIF(availability IS NOT NULL), COUNTIF(status_description IS NOT NULL)
FROM `{PROJECT_ID}.reporting.rpt_pitcher_season`
UNION ALL
SELECT 'rpt_batter_game', COUNT(*),
  COUNTIF(availability IS NOT NULL), COUNTIF(status_description IS NOT NULL)
FROM `{PROJECT_ID}.reporting.rpt_batter_game`
UNION ALL
SELECT 'rpt_pitcher_pitch_mix', COUNT(*),
  COUNTIF(availability IS NOT NULL), COUNTIF(status_description IS NOT NULL)
FROM `{PROJECT_ID}.reporting.rpt_pitcher_pitch_mix`
"""
status_attach_df = run_query(status_attach_sql)
print("Status attach coverage (NULL availability = played but not currently rostered):")
display(status_attach_df)

Status attach coverage (NULL availability = played but not currently rostered):


,model,n_rows,with_availability,with_status_desc
0,rpt_batter_season,530,521,521
1,rpt_pitcher_season,654,646,646
2,rpt_batter_game,17778,17631,17631
3,rpt_pitcher_pitch_mix,5886,5814,5814


## Spot-Checks  _(added 2026-06-21)_
Targeted checks that the status semantics are correct end-to-end.

In [23]:
# (a) A known IL pitcher reads the right status in rpt_pitcher_season.
# Graham Ashcraft (CIN) is on the 60-Day IL but has season stats -> his row
# should show those stats AND availability = "IL — 60-Day".
ashcraft_sql = f"""
SELECT pitcher_name, team_abbrev, games, availability, status_description
FROM `{PROJECT_ID}.reporting.rpt_pitcher_season`
WHERE LOWER(pitcher_name) LIKE '%ashcraft%'
ORDER BY games DESC
"""
ashcraft_df = run_query(ashcraft_sql)
print("Ashcraft spot-check (expect Graham -> 'IL — 60-Day'):")
display(ashcraft_df)

Ashcraft spot-check (expect Graham -> 'IL — 60-Day'):


,pitcher_name,team_abbrev,games,availability,status_description
0,"Ashcraft, Graham",CIN,25,IL — 60-Day,Injured 60-Day
1,"Ashcraft, Braxton",PIT,12,Active,Active


In [24]:
# (b) Attach-only proof: a 60-Day IL player present in dim_roster_status but
# ABSENT from rpt_batter_season. The stat models intentionally have NO null-stat
# row for him (grain unchanged) -- but he is fully recoverable from the
# dimension, so nobody is silently dropped. Power BI can LEFT JOIN per-visual.
missing_proof_sql = f"""
SELECT d.player_full_name, d.team_abbrev, d.availability
FROM `{PROJECT_ID}.marts.dim_roster_status` d
LEFT JOIN `{PROJECT_ID}.reporting.rpt_batter_season` b
  ON b.batter_id = d.mlbam_id
WHERE d.availability = 'IL — 60-Day' AND b.batter_id IS NULL
ORDER BY d.player_full_name
LIMIT 10
"""
missing_proof_df = run_query(missing_proof_sql)
print("60-Day IL players in the dimension but absent from rpt_batter_season")
print("(attach-only by design: no null rows in rpt_*, recoverable from the dim):")
display(missing_proof_df)
print(f"such players exist in the dimension: {len(missing_proof_df) > 0}")

60-Day IL players in the dimension but absent from rpt_batter_season
(attach-only by design: no null rows in rpt_*, recoverable from the dim):


,player_full_name,team_abbrev,availability
0,A.J. Puk,AZ,IL — 60-Day
1,AJ Smith-Shawver,ATL,IL — 60-Day
2,AJ Soldra,LAD,IL — 60-Day
3,Aaron Davenport,CLE,IL — 60-Day
4,Abraham Gomez,DET,IL — 60-Day
5,Abraham Negrin,CLE,IL — 60-Day
6,Adam Bloebaum,WSH,IL — 60-Day
7,Adam Laskey,COL,IL — 60-Day
8,Adam Mazur,MIA,IL — 60-Day
9,Adrian Quintana,SEA,IL — 60-Day


such players exist in the dimension: True


## Grain Guards  _(added 2026-06-21)_
Uniqueness guards on the new dimension keys, so this notebook would catch a
future fan-out (the same thing the dbt `unique` tests enforce on each run).

In [25]:
# dim_team is one row per team_id; dim_roster_status is one row per mlbam_id.
# dup_rows must be 0 for both.
grain_guard_sql = f"""
SELECT 'dim_team.team_id' AS grain_key,
  COUNT(*) AS n_rows, COUNT(DISTINCT team_id) AS distinct_keys,
  COUNT(*) - COUNT(DISTINCT team_id) AS dup_rows
FROM `{PROJECT_ID}.marts.dim_team`
UNION ALL
SELECT 'dim_roster_status.mlbam_id',
  COUNT(*), COUNT(DISTINCT mlbam_id),
  COUNT(*) - COUNT(DISTINCT mlbam_id)
FROM `{PROJECT_ID}.marts.dim_roster_status`
"""
grain_guard_df = run_query(grain_guard_sql)
print("Grain guards (dup_rows must be 0):")
display(grain_guard_df)
assert (grain_guard_df["dup_rows"] == 0).all(), "GRAIN VIOLATION: duplicate keys detected"
print("PASS: dim_team and dim_roster_status are unique on their grain keys.")

Grain guards (dup_rows must be 0):


,grain_key,n_rows,distinct_keys,dup_rows
0,dim_team.team_id,30,30,0
1,dim_roster_status.mlbam_id,8189,8189,0


PASS: dim_team and dim_roster_status are unique on their grain keys.
